# fMRIPrep: Preprocessing Functional MRI Data with Neurodesk on HPC

**Author**: Kelly G. Garner, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/garner-code"><img src="https://img.shields.io/badge/-Kelly_G._Garner-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

fMRIPrep is a robust preprocessing pipeline for functional MRI data that automates motion correction, registration to standard space, and quality assessment. This tutorial focuses on running fMRIPrep from Neurodesk, with particular attention to HPC-specific configuration and common pitfalls.

For a full executable walkthrough - downloading an open dataset and running fMRIPrep end-to-end - see the [fMRIPrep example notebook](../../examples/functional_imaging/fmriprep.ipynb).

:::{admonition} Learning Objectives
:class: tip

After completing this tutorial, you will be able to:

- Open the fMRIPrep container from the Neurodesk application menu
- Configure fMRIPrep with appropriate HPC resource parameters
- Avoid common pitfalls when running fMRIPrep on HPC systems
- Interpret and access quality control reports

:::

## Citation and Resources

**fMRIPrep**
: Esteban, O., et al. (2019). fMRIPrep: a robust preprocessing pipeline for functional MRI. *Nature Methods*, 16(1), 111–116. https://doi.org/10.1038/s41592-018-0235-4

**Official Documentation**
: https://fmriprep.org/

## Prerequisites

:::{admonition} Requirements
:class: warning

- Data must be in BIDS (Brain Imaging Data Structure) format
- Neurodesk running on your HPC system or local machine
- FreeSurfer license file (free to obtain from https://surfer.nmr.mgh.harvard.edu/registration.html)
- Sufficient disk space for preprocessing outputs (~50-100 GB per participant depending on sequence)
- Access to HPC job scheduler (SLURM, PBS, etc.) if running on HPC

:::

### Setup Checklist

- [ ] My data is in BIDS format
- [ ] Neurodesk is installed and running
- [ ] I have obtained a FreeSurfer license file
- [ ] I have placed the license file in `~/neurodesktop-storage/`
- [ ] I understand my HPC resource allocation limits
- [ ] I have sufficient disk space for outputs

## Section 1: Open fMRIPrep from Neurodesk

:::{note}
If this is your first time opening a tool from the Neurodesk application menu, see [Accessing Tools in Neurodesk - Section 3](../about_neurodesk/accessing_neurodesk_tools.ipynb) for a step-by-step walkthrough of launching the desktop and using the application menu.
:::

From the Neurodesktop application menu, navigate to:

**Applications → Neurodesk → Functional Imaging → fmriprep → fmriprep[version]**

![fMRIPrep in the Neurodesktrop applications menu](/static/tutorials/functional_imaging/fmriprep/fmriprep_menu.png)
*Select the fMRIPrep container.*

Choose the latest version available (at the bottom of the list). A terminal window will open inside the fMRIPrep container, ready for commands:

![fMRIPrep container terminal ready to use](/static/tutorials/functional_imaging/fmriprep/fmriprep_bash.png)
*The fMRIPrep container terminal open and ready to use.*

## Section 2: Run fMRIPrep

For a complete, executable example using an open dataset (Flanker task, OpenNeuro ds000102), see the [fMRIPrep example notebook](../../examples/functional_imaging/fmriprep.ipynb). The steps below cover the key configuration options for running on your own data.

### Setting thread and memory limits

Before running on HPC, limit ITK threads to match your job allocation:

```bash
export ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS=6
```

### Typical fMRIPrep command

```bash
fmriprep /path/to/bids_data \
         /path/to/bids_data/derivatives \
         participant \
         --fs-license-file ~/neurodesktop-storage/freesurfer.txt \
         --output-spaces T1w MNI152NLin2009cAsym fsaverage fsnative \
         --participant-label 01 \
         --nprocs 6 --mem 10000 \
         --skip_bids_validation \
         -v
```

### Key parameters

| Parameter | Purpose |
|-----------|---------|
| `--fs-license-file` | Path to your FreeSurfer license file (required) |
| `--output-spaces` | Spaces to resample outputs into (e.g. `T1w MNI152NLin2009cAsym fsaverage`) |
| `--participant-label` | One or more subject IDs to process (e.g. `01 02 03`) |
| `--nprocs` | CPU threads for fMRIPrep itself |
| `--mem` | Memory limit in MB (set to match your job allocation) |
| `-w` | Working directory for temporary files (use a large scratch partition on HPC) |
| `--skip_bids_validation` | Skip BIDS validation check (saves time if your dataset is already validated) |
| `--fs-subjects-dir` | Reuse an existing FreeSurfer subjects directory (avoids re-running recon-all) |

:::{note}
On a local machine (not HPC), you can omit `--nprocs` and `--mem`. fMRIPrep will use available system resources.
:::

## Section 3: Common Pitfalls and How to Avoid Them

### 1. Running out of disk space

fMRIPrep creates large temporary files during preprocessing (often several GB per subject). If your job hangs or crashes without a clear error:

- **Problem**: You ran out of space on the default scratch partition
- **Solution**: Redirect the working directory to a larger partition with `-w`:
  ```bash
  fmriprep /path/to/bids_data \
           /path/to/bids_data/derivatives \
           participant \
           -w /scratch/large_partition/fmriprep_work \
           --nprocs 6 --mem 10000
  ```
- **Best practice**: Always use a dedicated scratch directory on HPC. Clean up the work directory after successful runs - it can be many times larger than the final derivatives.

### 2. Parallel subject processing confusion

If you try to run multiple subjects simultaneously using a job scheduler (e.g. GNU Parallel or xargs pointing at the same working directory):

- **Problem**: fMRIPrep subjects can interfere with each other's temporary files and FreeSurfer outputs
- **Solution**: Either pass all subjects to a single fMRIPrep call (sequential processing), or give each job its own working directory:
  ```bash
  # Good: single call, sequential processing
  fmriprep ... --participant-label 01 02 03

  # Good: parallel jobs with isolated working directories
  fmriprep ... --participant-label 01 -w /scratch/work_sub01
  fmriprep ... --participant-label 02 -w /scratch/work_sub02
  ```
- **Credit**: Thanks to @thomshaw92 for identifying this issue.

### 3. Exceeding HPC resource limits

If your job is killed without an error message:

- **Problem**: Your job exceeded the CPU or memory limits set by the HPC scheduler
- **Solution**: Set `--nprocs` and `--mem` to match (or slightly under) your job allocation:
  ```bash
  # If you requested 6 CPUs and 10 GB RAM:
  fmriprep ... --nprocs 6 --mem 10000
  ```
- **Best practice**: Set these 10-20% below your actual allocation to allow headroom for OS and container overhead. Check resource usage with `top`, `htop`, or your HPC scheduler's monitoring tools during a test run.

### 4. FreeSurfer license errors

- **Problem**: fMRIPrep exits with `ERROR: license file /path/to/license not found` or similar
- **Solution**: Ensure the license file path passed to `--fs-license-file` exists and is readable inside the container. The `~/neurodesktop-storage/` directory is mounted into the container automatically.
  ```bash
  # Check the file exists before running:
  ls -lh ~/neurodesktop-storage/freesurfer.txt
  ```
- If you do not yet have a license, register for free at https://surfer.nmr.mgh.harvard.edu/registration.html.

### 5. Re-running after a crash

If fMRIPrep crashed partway through and you restart it:

- **Problem**: Incomplete FreeSurfer outputs in `--fs-subjects-dir` can cause fMRIPrep to skip recon-all silently but then fail on surface-based outputs
- **Solution**: Either delete the incomplete subject folder in the FreeSurfer subjects directory before rerunning, or use `--recon-all-args` to force re-running. Always check the fMRIPrep log for `[WARNING] Found existing FreeSurfer output` messages.

### 6. Viewing HTML QC reports on HPC

fMRIPrep generates a detailed HTML quality control report per subject in the derivatives folder (e.g. `derivatives/sub-01.html`):

- **Problem**: Viewing HTML reports through RDP (Remote Desktop Protocol) is slow and often crashes
- **Better options**:
  - Transfer the HTML file and its associated `sub-01.figures/` folder to your local machine and open it locally
  - Use the JupyterLab file browser (navigate to the derivatives folder and click the HTML file)
  - Use VNC instead of RDP for remote viewing
  - Mount the derivatives folder on your local machine via SFTP
- **Note**: The HTML report requires the accompanying `figures/` folder to be present - transfer both together.

## Summary

In this tutorial, you:

- Opened the fMRIPrep container from the Neurodesk application menu
- Configured fMRIPrep with standard options and HPC resource parameters
- Learned six common pitfalls and how to avoid them: disk space management, parallel-subject isolation, resource limit alignment, FreeSurfer license setup, crash recovery, and QC report access

For a complete end-to-end example with an open dataset, see the [fMRIPrep example notebook](../../examples/functional_imaging/fmriprep.ipynb).

:::{seealso}
- [fMRIPrep example notebook](../../examples/functional_imaging/fmriprep.ipynb) - Full executable walkthrough on the Flanker open dataset
- [MRIQC Tutorial](./mriqc.ipynb) - Quality control of MRI images before preprocessing
- [fMRIPrep Official Documentation](https://fmriprep.org/) - Full parameter reference and pipeline description
- [FreeSurfer License Registration](https://surfer.nmr.mgh.harvard.edu/registration.html) - Get your free FreeSurfer license
:::